In [1]:
from osxmetadata import OSXMetaData

from utils.paths import get_root_path

test_files_dir = get_root_path() / "notebooks" / "test-files"
txt_file_path = test_files_dir / "texts" / "txt" / "11.txt"
rtf_file_path = test_files_dir / "texts" / "rtf" / "165333.rtf"
csv_file_path = test_files_dir / "spreadsheets" / "csv" / "actions_under_antiquities_act.csv"
xlsx_file_path = test_files_dir / "spreadsheets" / "xls" / "1_NoIden.xlsx"
xlsm_file_path = test_files_dir / "spreadsheets" / "xls" / "61495-test.xlsm"
xls_file_path = test_files_dir / "spreadsheets" / "xls" / "3dFormulas.xls"
pptx_file_path = test_files_dir / "presentations" / "ppt" / "2411-Performance_Up.pptx"
ppt_file_path = test_files_dir / "presentations" / "ppt" / "23884_defense_FINAL_OOimport_edit.ppt"
pptm_file_path = test_files_dir / "presentations" / "ppt" / "PPTWithAttachments.pptm"
docm_file_path = test_files_dir / "documents" / "doc" / "45690.docm"
doc_file_path = test_files_dir / "documents" / "doc" / "47304.doc"
docx_file_path = test_files_dir / "documents" / "doc" / "52449.docx"

file_path_list = [
    {"path": txt_file_path, "ext": "txt"},
    {"path": rtf_file_path, "ext": "rtf"},
    {"path": csv_file_path, "ext": "csv"},
    {"path": xlsx_file_path, "ext": "xlsx"},
    {"path": xlsm_file_path, "ext": "xlsm"},
    {"path": xls_file_path, "ext": "xls"},
    {"path": pptx_file_path, "ext": "pptx"},
    {"path": ppt_file_path, "ext": "ppt"},
    {"path": pptm_file_path, "ext": "pptm"},
    {"path": docm_file_path, "ext": "docm"},
    {"path": doc_file_path, "ext": "doc"},
    {"path": docx_file_path, "ext": "docx"},
]

# 확장자별로 메타데이터 키 분류 및 보기 좋은 출력
from typing import Any, cast

ext_to_keys: dict[str, dict[str, list[str]]] = {}

# 값 존재 판정: None/빈 문자열/빈 컨테이너는 미존재로, False는 존재로 취급


from collections.abc import Sized


def _is_present_value(value: Any) -> bool:
    if value is None:
        return False
    if isinstance(value, bool):
        return True
    if isinstance(value, (str, bytes)):
        try:
            s = value.decode("utf-8", errors="ignore") if isinstance(value, bytes) else value
        except Exception:
            s = str(value)
        return len(s) > 0
    if isinstance(value, Sized):
        return len(value) > 0
    return True


for file_path in file_path_list:
    path = file_path["path"]
    ext: str = cast(str, file_path["ext"])

    # dict로 변환
    metadata = OSXMetaData(str(path)).asdict()
    md_dict: dict[str, Any] = {str(k): v for k, v in metadata.items()}

    present_keys: list[str] = sorted([k for k, v in md_dict.items() if _is_present_value(v)])
    none_keys: list[str] = sorted([k for k, v in md_dict.items() if not _is_present_value(v)])

    ext_to_keys[ext] = {"present": present_keys, "none": none_keys}


def _format_key_list(keys: list[str]) -> str:
    if not keys:
        return "(none)"
    return ", ".join(keys)


# 공통 present 키와 확장자별 유니크 present 키 계산


def _intersect_sets_str(sets: list[set[str]]) -> set[str]:
    if not sets:
        return set()
    result: set[str] = set(sets[0])
    for s in sets[1:]:
        result = result.intersection(s)
    return result


ext_to_present: dict[str, set[str]] = {
    ext: set(groups["present"]) for ext, groups in ext_to_keys.items()
}
all_exts: list[str] = sorted(list(ext_to_keys.keys()))
present_intersection: set[str] = _intersect_sets_str([ext_to_present[e] for e in all_exts])

# 키 빈도 집계(각 키가 몇 개 확장자에서 present인지)
key_to_count: dict[str, int] = {}
if all_exts:
    for ext in all_exts:
        for k in ext_to_present[ext]:
            key_to_count[k] = key_to_count.get(k, 0) + 1

# 확장자별 추가/유니크/공유-비공통 키 계산
extra_by_ext: dict[str, list[str]] = {}
unique_by_ext: dict[str, list[str]] = {}
shared_noncommon_by_ext: dict[str, list[str]] = {}
num_exts: int = len(all_exts)
if all_exts:
    for ext in all_exts:
        extra_set: set[str] = ext_to_present[ext] - present_intersection
        unique_set: set[str] = {k for k in extra_set if key_to_count.get(k, 0) == 1}
        shared_noncommon_set: set[str] = {
            k for k in extra_set if 1 < key_to_count.get(k, 0) < num_exts
        }
        extra_by_ext[ext] = sorted(list(extra_set))
        unique_by_ext[ext] = sorted(list(unique_set))
        shared_noncommon_by_ext[ext] = sorted(list(shared_noncommon_set))

# 공통 present 키 출력
print("=== COMMON (present in all) ===")
print(f"- Count: {len(present_intersection)}")
print(f"- Keys: {_format_key_list(sorted(list(present_intersection)))}")
print()

# 확장자별 요약 출력
for ext in all_exts:
    present = ext_to_keys[ext]["present"]
    none = ext_to_keys[ext]["none"]
    total = len(present) + len(none)
    unique = unique_by_ext.get(ext, [])

    print(f"=== .{ext} ===")
    print(f"- Total keys: {total}")
    print(f"- Present ({len(present)}): {_format_key_list(present)}")
    print(f"- None ({len(none)}): {_format_key_list(none)}")
    extra = extra_by_ext.get(ext, [])
    shared_noncommon = shared_noncommon_by_ext.get(ext, [])
    print(f"- Extra over common ({len(extra)}): {_format_key_list(extra)}")
    print(f"- Unique to .{ext} ({len(unique)}): {_format_key_list(unique)}")
    print(f"- Shared non-common ({len(shared_noncommon)}): {_format_key_list(shared_noncommon)}")
    print()

=== COMMON (present in all) ===
- Count: 20
- Keys: findercolor, kMDItemAttributeChangeDate, kMDItemContentCreationDate, kMDItemContentModificationDate, kMDItemContentType, kMDItemContentTypeTree, kMDItemDateAdded, kMDItemDisplayName, kMDItemFSContentChangeDate, kMDItemFSCreationDate, kMDItemFSInvisible, kMDItemFSIsExtensionHidden, kMDItemFSLabel, kMDItemFSName, kMDItemFSOwnerGroupID, kMDItemFSOwnerUserID, kMDItemFSSize, kMDItemKind, kMDItemPath, stationerypad

=== .csv ===
- Total keys: 185
- Present (20): findercolor, kMDItemAttributeChangeDate, kMDItemContentCreationDate, kMDItemContentModificationDate, kMDItemContentType, kMDItemContentTypeTree, kMDItemDateAdded, kMDItemDisplayName, kMDItemFSContentChangeDate, kMDItemFSCreationDate, kMDItemFSInvisible, kMDItemFSIsExtensionHidden, kMDItemFSLabel, kMDItemFSName, kMDItemFSOwnerGroupID, kMDItemFSOwnerUserID, kMDItemFSSize, kMDItemKind, kMDItemPath, stationerypad
- None (165): _kMDItemUserTags, kMDItemAcquisitionMake, kMDItemAcquisition

In [ ]:
from pprint import pprint

for file_path in file_path_list:
    path = file_path["path"]
    ext: str = cast(str, file_path["ext"])

    # dict로 변환
    metadata = OSXMetaData(str(path)).asdict()
    md_dict: dict[str, Any] = {str(k): v for k, v in metadata.items()}

    present_keys: list[str] = sorted([k for k, v in md_dict.items() if _is_present_value(v)])
    # none_keys: list[str] = sorted([k for k, v in md_dict.items() if not _is_present_value(v)])

    print(f"======= {ext} =======")
    present_dict = {k: md_dict[k] for k in present_keys}
    ext_to_keys[ext] = {"present": present_dict}
    pprint(ext_to_keys[ext])

======= txt =======
{'present': {'findercolor': 0,
             'kMDItemAttributeChangeDate': datetime.datetime(2025, 8, 31, 18, 48, 51, 715086),
             'kMDItemContentCreationDate': datetime.datetime(2025, 8, 30, 9, 40, 37, 322077),
             'kMDItemContentModificationDate': datetime.datetime(2025, 8, 30, 9, 40, 37, 322552),
             'kMDItemContentType': 'public.plain-text',
             'kMDItemContentTypeTree': ['public.plain-text',
                                        'public.text',
                                        'public.data',
                                        'public.item',
                                        'public.content'],
             'kMDItemDateAdded': datetime.datetime(2025, 8, 30, 9, 40, 37, 322077),
             'kMDItemDisplayName': '11.txt',
             'kMDItemFSContentChangeDate': datetime.datetime(2025, 8, 30, 9, 40, 37, 322552),
             'kMDItemFSCreationDate': datetime.datetime(2025, 8, 30, 9, 40, 37, 322077),
         

# 기본 공통 키

- `findercolor`: Finder 색상
- `kMDItemAttributeChangeDate`: 파일 메타데이터 속성 변경 시간(ctime). 예) 이름 변경
- `kMDItemContentCreationDate`: 콘텐츠 생성 시간. EXIF/문서 내부 Created가 있으면 그 값, 없으면 생성일로 폴백. [v]
- `kMDItemContentModificationDate`: 콘텐츠 논리적 수정 시간. 내부 LastModified가 있으면 그 값, 없으면 mtime으로 폴백. [v]
- `kMDItemContentType`: Uniform Type Identifier, UTI값. 예) public.jpeg, com.adobe.pdf... [v]
- `kMDItemContentTypeTree`: UTI 트리
- `kMDItemDateAdded`: Finder에 추가된 날짜 [v]
- `kMDItemDisplayName`: Finder에 표시되는 이름. i18n,확장자 숨김 반영
- `kMDItemFSContentChangeDate`: 파일 내용 변경시간(mtime). 바이트 내용 변경시 갱신 [v]
- `kMDItemFSCreationDate`: 파일 생성 시간 [v]
- `kMDItemFSInvisible`: 파일 숨김 여부 [v]
- `kMDItemFSIsExtensionHidden`: 확장자 숨김 여부(시스템 설정)
- `kMDItemFSLabel`: Finder 라벨 인덱스. 색상명은 findercolor에 매핑 가능
- `kMDItemFSName`: 파일의 원래 이름. i18n 미적용, 확장자 포함 [v]
- `kMDItemFSOwnerGroupID`: 소유 그룹 GID
- `kMDItemFSOwnerUserID`: 소유 사용자 UID
- `kMDItemFSSize`: 파일 크기(바이트) [v]
- `kMDItemKind`: 인간 친화적 파일 분류 문자열(현지화됨) [v]
- `kMDItemPath`: 절대 경로 [v]
- `stationerypad`: Stationary Pad 선택 여부. 열 때 복사본을 생성하는 플래그

## 추가 공통 키: 상태에 따라 나타남

- `kMDItemLastUsedDate`: 파일 마지막 실행 시간. 파일 생성 후 실행하지 않았으면 None임 [v]
- `kMDItemWhereFroms`: 다운로드/복사 출처 URL [v]
- `kMDItemFinderComment`: Finder 파일 코멘트 [v]
- `_kMDItemUserTags`: Finder 파일 태그 [v]
- `kMDItemDownloadedDate`: 다운로드된 시각

# 정규화

- 인덱싱 등 활용을 용이하게 하기 위해 속성 키 값을 정규화

## 메타데이터 키 후보 (순서 != 순위)

- `kMDItemPath` => `path`
- `kMDItemFSName` => `name_full`
- `kMDItemFSSize` => `size`
- `kMDItemFSCreationDate` => `creation_date`
- `kMDItemFSContentChangeDate` => `modification_date`
- `kMDItemContentCreationDate` => `content_creation_date`
- `kMDItemContentModificationDate` => `content_modification_date`
- `kMDItemDateAdded` => `added_date`
- `kMDItemLastUsedDate` => `last_used_date`
- `_kMDItemUserTags` => `finder_tags`
- `kMDItemContentType` => `uniform_type_identifier`
- `kMDItemKind` => `file_kind`
- `kMDItemWhereFroms` => `where_from`
- `kMDItemFSInvisible` => `is_invisible`

## 파생 키

- `name_stem`: 확장자 제거된 이름 <= `kMDItemFSName`
- `extension`: 확장자 <= `kMDItemFSName`
- `depth`: 파일 경로 깊이 <= `kMDItemPath`
- `dir_path`: 파일 경로가 제외된 디렉토리 경로. 예) `/Users/foo/Downloads` <= `kMDItemPath`
- `parent_dir_name`: 부모 폴더 이름 <= `kMDItemPath`

## 추가 고려할 키

데이터클래스 속성으로 추가할 가능성이 있는 키들. 사용하는데 있어 유용성 및 비용 등을 판단해서 결정

- `sha256`: 파일 내용 읽어서 hash화. 파일 내용의 중복을 체크하기 위함. 중복 인덱싱, 임베딩 방지
- `mime`: <= `kMDItemContentType` pyobjc UniformTypeIdentifiers framework의 `UTType`으로 정확한 매핑 가능
- `encoding`: 파일을 읽어서 encoding 값을 추론. TextLoader에서 필요. 그러나 TextLoader에 encoding을 자동 감지하는 플래그(`autodetect_encoding`)와 메서드 내장되어 있어서 불필요할 것 같음


In [ ]:
from pathlib import Path

metadata = OSXMetaData(str(txt_file_path))
file_stat = txt_file_path.stat()


def _calculate_depth_from_home(file_path: Path) -> int:
    """HOME 디렉토리로부터의 깊이를 계산합니다."""
    home_path = Path.home()
    try:
        relative_path = file_path.relative_to(home_path)
        return len(relative_path.parts) - 1  # 파일 자체는 제외하고 디렉토리 깊이만 계산
    except ValueError:
        # 파일이 HOME 디렉토리 하위에 없는 경우
        return -1


voyager_file_record_test = {
    "location_key": f"{file_stat.st_dev}-{file_stat.st_ino}",
    "name_full": txt_file_path.name,
    "name_stem": txt_file_path.stem,
    "extension": txt_file_path.suffix,
    "size_from_stat": file_stat.st_size,  # int
    "size_from_metadata": metadata["kMDItemFSSize"],  # float/
    "uniform_type_identifier": metadata.get("kMDItemContentType"),
    "file_kind": metadata.get("kMDItemKind"),
    "path": txt_file_path,
    "dir_path": txt_file_path.parent,
    "parent_dir_name": txt_file_path.parent.name,
    "depth": _calculate_depth_from_home(txt_file_path),
    "creation_date_from_stat": file_stat.st_ctime,  # float
    "creation_date_from_metadata": metadata.get("kMDItemFSCreationDate"),  # datetime
    "modification_date_from_stat": file_stat.st_mtime,  # float
    "modification_date_from_metadata": metadata.get("kMDItemFSContentChangeDate"),  # datetime
    "content_creation_date_from_metadata": metadata.get("kMDItemContentCreationDate"),  # datetime
    "content_modification_date_from_metadata": metadata.get(
        "kMDItemContentModificationDate"
    ),  # datetime
    "added_date": metadata.get("kMDItemDateAdded"),
    "last_used_date": metadata.get("kMDItemLastUsedDate"),
    "finder_tags": metadata.get("_kMDItemUserTags"),
    "where_from": metadata.get("kMDItemWhereFroms"),
    "is_invisible": metadata.get("kMDItemFSInvisible"),
    "sha256": "",
    "original_metadata": metadata.to_json(),
}

voyager_file_record_test

{'location_key': '16777233-75967536',
 'name_full': '11.txt',
 'name_stem': '11',
 'extension': '.txt',
 'size_from_stat': 174355,
 'size_from_metadata': 174355.0,
 'uniform_type_identifier': 'public.plain-text',
 'file_kind': 'Plain Text Document',
 'path': PosixPath('/Users/jongmin/Desktop/voyager-fm/voyager-app-backend/notebooks/test-files/texts/txt/11.txt'),
 'dir_path': PosixPath('/Users/jongmin/Desktop/voyager-fm/voyager-app-backend/notebooks/test-files/texts/txt'),
 'parent_dir_name': 'txt',
 'depth': 7,
 'creation_date_from_stat': 1756612047.7033691,
 'creation_date_from_metadata': datetime.datetime(2025, 8, 30, 9, 40, 37, 322077),
 'modification_date_from_stat': 1756514437.322552,
 'modification_date_from_metadata': datetime.datetime(2025, 8, 30, 9, 40, 37, 322552),
 'content_creation_date_from_metadata': datetime.datetime(2025, 8, 30, 9, 40, 37, 322077),
 'content_modification_date_from_metadata': datetime.datetime(2025, 8, 30, 9, 40, 37, 322552),
 'added_date': datetime.date

In [ ]:
from dataclasses import dataclass
from datetime import datetime
from pathlib import Path
from typing import Any, Optional


def _calculate_depth_from_home(file_path: Path) -> int:
    """HOME 디렉토리로부터의 깊이를 계산합니다."""
    home_path = Path.home()
    try:
        relative_path = file_path.relative_to(home_path)
        return len(relative_path.parts) - 1  # 파일 자체는 제외하고 디렉토리 깊이만 계산
    except ValueError:
        # 파일이 HOME 디렉토리 하위에 없는 경우
        return -1


def _get_str(key: str) -> Optional[str]:
    v_obj: object | None = metadata.get(key)
    if type(v_obj) is str:
        return v_obj
    if type(v_obj) is list:
        return v_obj[0] if v_obj else None
    return None


def _get_dt(key: str, fallback_ts: float) -> datetime:
    v = metadata.get(key)
    if isinstance(v, datetime):
        return v
    return datetime.fromtimestamp(fallback_ts)


def _get_dt_opt(key: str) -> Optional[datetime]:
    v = metadata.get(key)
    return v if isinstance(v, datetime) else None


def _get_tags(key: str) -> list[str]:
    v = metadata.get(key)
    if type(v) is list:
        try:
            return [str(x) for x in v]
        except Exception:
            return []
    return []


def _get_bool(key: str, default: bool = False) -> bool:
    v = metadata.get(key)
    return v if type(v) is bool else default


@dataclass(frozen=True, slots=True)
class VoyagerFileMeta:
    name_full: str
    name_stem: str
    extension: str
    size: int
    uniform_type_identifier: Optional[str]
    file_kind: Optional[str]
    path: Path
    dir_path: Path
    parent_dir_name: str
    depth: int
    creation_date: datetime
    modification_date: datetime
    content_creation_date: datetime
    content_modification_date: datetime
    added_date: datetime
    last_used_date: Optional[datetime]
    finder_tags: list[str]
    is_invisible: bool
    where_from: Optional[str]
    # sha256: str
    original_metadata: dict[str, Any]

    @classmethod
    def from_file_path(cls, file_path: Path) -> "VoyagerFileMeta":
        stat = file_path.stat()
        metadata = OSXMetaData(str(file_path))

        return cls(
            name_full=file_path.name,
            name_stem=file_path.stem,
            extension=file_path.suffix,
            size=stat.st_size,
            uniform_type_identifier=_get_str("kMDItemContentType"),
            file_kind=_get_str("kMDItemKind"),
            path=file_path,
            dir_path=file_path.parent,
            parent_dir_name=file_path.parent.name,
            depth=_calculate_depth_from_home(file_path),
            creation_date=_get_dt("kMDItemFSCreationDate", stat.st_ctime),
            modification_date=_get_dt("kMDItemFSContentChangeDate", stat.st_mtime),
            content_creation_date=_get_dt("kMDItemContentCreationDate", stat.st_ctime),
            content_modification_date=_get_dt("kMDItemContentModificationDate", stat.st_mtime),
            added_date=_get_dt("kMDItemDateAdded", stat.st_ctime),
            last_used_date=_get_dt_opt("kMDItemLastUsedDate"),
            finder_tags=_get_tags("_kMDItemUserTags"),
            is_invisible=_get_bool("kMDItemFSInvisible", False),
            where_from=_get_str("kMDItemWhereFroms"),
            original_metadata=metadata.asdict(),
        )

In [ ]:
from core.file_crawler.extractor import walk_files_concurrently

for file_path in walk_files_concurrently(test_files_dir):
    print(file_path)
    VoyagerFileMeta.from_file_path(file_path)


(PosixPath('/Users/jongmin/Desktop/voyager-fm/voyager-app-backend/notebooks/test-files/.DS_Store'), os.stat_result(st_mode=33188, st_ino=75833604, st_dev=16777233, st_nlink=1, st_uid=501, st_gid=20, st_size=8196, st_atime=1756520339, st_mtime=1756631769, st_ctime=1756631769))
(PosixPath('/Users/jongmin/Desktop/voyager-fm/voyager-app-backend/notebooks/test-files/presentations/.DS_Store'), os.stat_result(st_mode=33188, st_ino=75667924, st_dev=16777233, st_nlink=1, st_uid=501, st_gid=20, st_size=6148, st_atime=1756477956, st_mtime=1756480330, st_ctime=1756480330))
(PosixPath('/Users/jongmin/Desktop/voyager-fm/voyager-app-backend/notebooks/test-files/texts/.DS_Store'), os.stat_result(st_mode=33188, st_ino=75860340, st_dev=16777233, st_nlink=1, st_uid=501, st_gid=20, st_size=6148, st_atime=1756612023, st_mtime=1756631769, st_ctime=1756631769))
(PosixPath('/Users/jongmin/Desktop/voyager-fm/voyager-app-backend/notebooks/test-files/documents/pdf/PDF document.pdf'), os.stat_result(st_mode=33188

In [1]:
import logging
from pathlib import Path
from typing import Generator, Iterable, Literal, Optional

from core.file_crawler import VoyagerFileMeta, walk_files_concurrently

_logger = logging.getLogger("voyager.pipeline")
if not _logger.handlers:
    logging.basicConfig(level=logging.INFO)


def to_voyager_file_meta(
    items: Iterable[Path],
    *,
    on_error: Literal["log", "skip", "raise"] = "log",
    logger: Optional[logging.Logger] = None,
) -> Generator[VoyagerFileMeta, None, None]:
    """Path → VoyagerFileMeta 제너레이터 변환.

    - 입력: Iterable[Path]. 디렉토리는 조용히 스킵합니다.
    - on_error 정책:
      - "log": FileNotFoundError/PermissionError는 warning, 그 외 예외는 error(exc_info=True)로 로깅 후 스킵
      - "skip": 로깅 없이 스킵
      - "raise": 예외를 전파하여 스트림을 중단
    - logger: 사용자 지정 로거 사용 가능(미지정 시 "voyager.pipeline").
    - 반환: `VoyagerFileMeta` 항목을 순차적으로 생성하는 제너레이터.
    """
    log = logger or _logger
    for path in items:
        try:
            yield VoyagerFileMeta.from_file_path(path)
        except (FileNotFoundError, PermissionError) as e:
            if on_error == "raise":
                raise
            if on_error == "log":
                log.warning("Skip %s (%s): %s", path, e.__class__.__name__, e)
            continue
        except Exception as e:
            if on_error == "raise":
                raise
            if on_error == "log":
                log.error("Error processing %s: %s", path, e, exc_info=True)
            continue


def walk_voyager_file_records(
    root: Path,
    *,
    on_error: Literal["log", "skip", "raise"] = "log",
    logger: Optional[logging.Logger] = None,
) -> Generator[VoyagerFileMeta, None, None]:
    """디렉토리를 동시 스캔해 VoyagerFileMeta 스트림을 생성."""
    yield from to_voyager_file_meta(walk_files_concurrently(root), on_error=on_error, logger=logger)


# 실행 예시
from utils.paths import get_root_path

root = get_root_path() / "notebooks" / "test-files"

stream = walk_voyager_file_records(root, on_error="log")
for rec in stream:
    print(rec.path, rec.uniform_type_identifier)


/Users/jongmin/Desktop/voyager-fm/voyager-app-backend/notebooks/test-files/.DS_Store dyn.ah62d4rv4gk8wakbaea
/Users/jongmin/Desktop/voyager-fm/voyager-app-backend/notebooks/test-files/texts/.DS_Store dyn.ah62d4rv4gk8wakbaea
/Users/jongmin/Desktop/voyager-fm/voyager-app-backend/notebooks/test-files/presentations/.DS_Store dyn.ah62d4rv4gk8wakbaea
/Users/jongmin/Desktop/voyager-fm/voyager-app-backend/notebooks/test-files/spreadsheets/xls/Booleans.xlsx org.openxmlformats.spreadsheetml.sheet
/Users/jongmin/Desktop/voyager-fm/voyager-app-backend/notebooks/test-files/spreadsheets/xls/excelant.xls com.microsoft.excel.xls
/Users/jongmin/Desktop/voyager-fm/voyager-app-backend/notebooks/test-files/spreadsheets/xls/53282b.xlsx org.openxmlformats.spreadsheetml.sheet
/Users/jongmin/Desktop/voyager-fm/voyager-app-backend/notebooks/test-files/spreadsheets/xls/63934.xlsx org.openxmlformats.spreadsheetml.sheet
/Users/jongmin/Desktop/voyager-fm/voyager-app-backend/notebooks/test-files/spreadsheets/xls/li

In [ ]:
from itertools import islice

from core.file_crawler import walk_files_concurrently
from core.file_crawler.extractor import to_voyager_file_meta
from utils.paths import get_root_path

root = get_root_path() / "notebooks" / "test-files"
gen_files = walk_files_concurrently(root)
for i in islice(to_voyager_file_meta(gen_files), 10):
    print(i)


VoyagerFileMeta(name_full='.DS_Store', name_stem='.DS_Store', extension='', size=8196, uniform_type_identifier='dyn.ah62d4rv4gk8wakbaea', file_kind='Document', path=PosixPath('/Users/jongmin/Desktop/voyager-fm/voyager-app-backend/notebooks/test-files/.DS_Store'), dir_path=PosixPath('/Users/jongmin/Desktop/voyager-fm/voyager-app-backend/notebooks/test-files'), parent_dir_name='test-files', depth=5, creation_date=datetime.datetime(2025, 8, 29, 23, 42, 59, 238230), modification_date=datetime.datetime(2025, 8, 31, 18, 16, 9, 334268), content_creation_date=datetime.datetime(2025, 8, 29, 23, 42, 59, 238230), content_modification_date=datetime.datetime(2025, 8, 31, 18, 16, 9, 334268), added_date=datetime.datetime(2025, 8, 31, 18, 16, 9, 334268), last_used_date=datetime.datetime(2025, 8, 31, 18, 16, 9, 334268), finder_tags=[], is_invisible=True, where_from=None, original_metadata={'kMDItemFSContentChangeDate': datetime.datetime(2025, 8, 31, 18, 16, 9, 334268), 'kMDItemTheme': None, 'kMDLabelBu